In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os

# Carpeta donde vivirán TODOS los audios para siempre
CARPETA_AUDIOS = "/content/drive/MyDrive/manim_audios"
os.makedirs(CARPETA_AUDIOS, exist_ok=True)
print(f"✅ Carpeta lista: {CARPETA_AUDIOS}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Carpeta lista: /content/drive/MyDrive/manim_audios


In [2]:
# ============================================================
# INSTALACIÓN
# ============================================================
!pip install -q google-genai pydub
!apt-get install -q ffmpeg

# ============================================================
# IMPORTACIONES
# ============================================================
import io
import os
import wave
import time
import json
from google import genai
from google.genai import types
from pydub import AudioSegment
from google.colab import userdata

# ============================================================
# API KEY
# ============================================================
try:
    API_KEY = userdata.get("GOOGLE_API_KEY")
    print("✅ API Key cargada.")
except Exception:
    raise ValueError("❌ No se encontró GOOGLE_API_KEY en Secrets.")

# ============================================================
# CARPETA DE DESTINO EN DRIVE
# ============================================================
CARPETA_AUDIOS = "/content/drive/MyDrive/manim_audios"
os.makedirs(CARPETA_AUDIOS, exist_ok=True)

# ============================================================
# FUNCIÓN PRINCIPAL — guarda en Drive y retorna bytes + ruta
# ============================================================
def retornar_MP3(
    texto: str,
    api_key: str,
    nombre_archivo: str,              # ej: "audio_escena_1_intro.mp3"
    carpeta: str = CARPETA_AUDIOS,    # destino en Drive
    forzar: bool = False,             # True = regenerar aunque exista
    max_reintentos: int = 4,
    espera_inicial: float = 3.0,
) -> dict:
    """
    Convierte texto a MP3, guarda en Drive y en disco local.

    Retorna un dict con:
        - bytes     : contenido del MP3
        - ruta_drive: ruta completa en Drive
        - ruta_local: ruta en disco local de Colab
        - duracion  : duración en segundos
        - desde_cache: True si se reutilizó un archivo existente
    """

    ruta_drive = os.path.join(carpeta, nombre_archivo)
    ruta_local = f"/content/{nombre_archivo}"

    # ── Si ya existe en Drive y no se fuerza regeneración ────────────
    if os.path.exists(ruta_drive) and not forzar:
        import shutil
        shutil.copy(ruta_drive, ruta_local)
        seg      = AudioSegment.from_file(ruta_drive, format="mp3")
        duracion = len(seg) / 1000.0
        print(f"   ♻️  Reutilizado desde Drive: {nombre_archivo} "
              f"({duracion:.2f}s)")
        with open(ruta_local, "rb") as f:
            mp3_bytes = f.read()
        return {
            "bytes"      : mp3_bytes,
            "ruta_drive" : ruta_drive,
            "ruta_local" : ruta_local,
            "duracion"   : duracion,
            "desde_cache": True,
        }

    # ── Generar nuevo audio ───────────────────────────────────────────
    print(f"   🎙️  Generando: {nombre_archivo}")
    client = genai.Client(api_key=api_key)

    ESTILO = (
        "Habla con voz profesional, pausada, muy emotiva "
        "y con acento paisa de Medellín, Colombia. "
        "Texto a leer:\n\n"
    )

    config_tts = types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                    voice_name="Puck"
                )
            )
        ),
    )

    response     = None
    ultimo_error = None

    for intento in range(1, max_reintentos + 1):
        try:
            print(f"      🔄 Intento {intento}/{max_reintentos}...")
            response = client.models.generate_content(
                model="gemini-2.5-flash-preview-tts",
                contents=ESTILO + texto,
                config=config_tts,
            )
            print(f"      ✅ Respuesta recibida.")
            break
        except Exception as e:
            ultimo_error = e
            msg = str(e)
            es_transitorio = any(
                c in msg for c in ["500", "503", "INTERNAL", "UNAVAILABLE"]
            )
            if es_transitorio and intento < max_reintentos:
                espera = espera_inicial * (2 ** (intento - 1))
                print(f"      ⚠️  Error transitorio. Esperando {espera:.0f}s...")
                time.sleep(espera)
            else:
                raise RuntimeError(
                    f"❌ Falló tras {intento} intento(s): {ultimo_error}"
                ) from e

    # ── Validar audio ─────────────────────────────────────────────────
    try:
        audio_data = (response.candidates[0]
                              .content.parts[0]
                              .inline_data.data)
        assert audio_data
    except (AttributeError, IndexError, AssertionError) as e:
        raise RuntimeError(f"❌ Respuesta sin audio: {e}") from e

    # ── PCM → WAV → MP3 ───────────────────────────────────────────────
    wav_buffer = io.BytesIO()
    with wave.open(wav_buffer, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(24000)
        wf.writeframes(audio_data)
    wav_buffer.seek(0)

    mp3_buffer = io.BytesIO()
    AudioSegment.from_wav(wav_buffer).export(
        mp3_buffer, format="mp3", bitrate="192k"
    )
    mp3_bytes = mp3_buffer.getvalue()

    # ── Guardar en Drive (permanente) Y en disco local (para Manim) ───
    with open(ruta_drive, "wb") as f:
        f.write(mp3_bytes)
    with open(ruta_local, "wb") as f:
        f.write(mp3_bytes)

    duracion = len(AudioSegment.from_file(
        io.BytesIO(mp3_bytes), format="mp3"
    )) / 1000.0

    print(f"      💾 Guardado en Drive: {ruta_drive}")
    print(f"      ⏱️  Duración: {duracion:.2f}s  |  "
          f"Tamaño: {len(mp3_bytes)/1024:.1f} KB")

    return {
        "bytes"      : mp3_bytes,
        "ruta_drive" : ruta_drive,
        "ruta_local" : ruta_local,
        "duracion"   : duracion,
        "desde_cache": False,
    }

print("✅ Función retornar_MP3 lista.")

E: Could not get lock /var/lib/dpkg/lock. It is held by process 10889 (dpkg)
E: Unable to lock the administration directory (/var/lib/dpkg/), is another process using it?
✅ API Key cargada.
✅ Función retornar_MP3 lista.


In [3]:
# ── Guiones ───────────────────────────────────────────────────────────
GUIONES = [
    ("audio_escena_1_intro.mp3",
     "Hola, y bienvenidos. En este video exploraremos paso a paso "
     "las diferentes transformaciones gráficas de la función "
     "exponencial básica: y igual a e elevada a x."),
    ("audio_escena_2_base.mp3",
     "Comencemos con nuestra función base. Observen su curva "
     "característica y su punto clave en cero coma uno."),
    ("audio_escena_3_caso1.mp3",
     "Caso uno: Desplazamiento vertical. Restamos dos unidades "
     "y la gráfica desciende. El punto pasa de cero uno a cero menos uno."),
    ("audio_escena_4_caso2.mp3",
     "Caso dos: Desplazamiento horizontal. La gráfica se mueve "
     "dos unidades a la derecha. El punto queda en dos coma uno."),
    ("audio_escena_5_caso3.mp3",
     "Caso tres: Reflexión sobre el eje x. La curva se invierte "
     "hacia abajo. El punto pasa a cero coma menos uno."),
    ("audio_escena_6_caso4.mp3",
     "Caso cuatro: Reflexión sobre el eje y. La gráfica se invierte "
     "lateralmente. El punto permanece en cero coma uno."),
    ("audio_escena_7_caso5.mp3",
     "Caso cinco: Doble reflexión. Aplicamos negativo a la función "
     "y al exponente. La curva cambia de cuadrante y dirección."),
    ("audio_escena_8_caso6.mp3",
     "Caso seis: Reflexión sobre y igual a cuatro. El punto sube "
     "simétricamente hasta cero coma siete."),
    ("audio_escena_9_caso7.mp3",
     "Caso siete: Reflexión sobre x igual a dos. El punto se traslada "
     "horizontalmente hasta cuatro coma uno."),
    ("audio_escena_10_cierre.mp3",
     "Hemos completado las siete transformaciones de la función "
     "exponencial. Espero que este recorrido visual haya sido útil. "
     "Hasta la próxima clase."),
]

# ── Control de rate limit ─────────────────────────────────────────────
REQUESTS_POR_MINUTO = 3
ESPERA_ENTRE_LOTES  = 90   # segundos
ESPERA_POR_429      = 90

DURACIONES  = {}
lote_actual = 0

def esperar(segundos, motivo="Esperando"):
    for s in range(segundos, 0, -1):
        print(f"   ⏳ {motivo}: {s:3d}s...", end="\r")
        time.sleep(1)
    print(f"   ✅ {motivo}: listo.          ")

print("🚀 Iniciando generación de audios...\n")
print(f"   Carpeta destino: {CARPETA_AUDIOS}\n")

for i, (nombre, texto) in enumerate(GUIONES):

    # Pausa proactiva cada 3 requests nuevos
    if lote_actual == REQUESTS_POR_MINUTO:
        print(f"\n⏸️  Límite de {REQUESTS_POR_MINUTO}/min alcanzado.")
        esperar(ESPERA_ENTRE_LOTES, "Pausa proactiva")
        lote_actual = 0

    print(f"\n📢 [{i+1}/{len(GUIONES)}] {nombre}")

    # Reintento ante 429
    for intento in range(1, 7):
        try:
            resultado = retornar_MP3(texto, API_KEY, nombre)
            DURACIONES[nombre] = resultado["duracion"]
            # Solo cuenta como request nuevo si no vino del caché
            if not resultado["desde_cache"]:
                lote_actual += 1
                # Pausa cortés entre requests del mismo lote
                if lote_actual < REQUESTS_POR_MINUTO and i < len(GUIONES)-1:
                    esperar(5, "Pausa entre requests")
            break

        except Exception as e:
            if ("429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)) \
                    and intento < 6:
                espera = ESPERA_POR_429 + (30 * (intento - 1))
                print(f"\n   🚫 429 — intento {intento}/6")
                esperar(espera, "Pausa por rate limit")
            else:
                print(f"\n   ❌ Error definitivo: {e}")
                raise

# ── Guardar JSON ──────────────────────────────────────────────────────
json.dump(DURACIONES, open("duraciones.json", "w"), indent=2)

print("\n" + "═"*55)
print("🎉 TODOS LOS AUDIOS LISTOS:")
total = 0
for nombre, dur in DURACIONES.items():
    icono = "♻️ " if retornar_MP3.__doc__ and os.path.exists(
        f"/content/{nombre}") else "✅"
    print(f"   ✅  {nombre:<35}  {dur:.2f}s")
    total += dur
print(f"\n   ⏱️  Duración total: {total:.1f}s ({total/60:.1f} min)")
print(f"   💾 Carpeta Drive:  {CARPETA_AUDIOS}")
print("═"*55)


🚀 Iniciando generación de audios...

   Carpeta destino: /content/drive/MyDrive/manim_audios


📢 [1/10] audio_escena_1_intro.mp3
   ♻️  Reutilizado desde Drive: audio_escena_1_intro.mp3 (15.25s)

📢 [2/10] audio_escena_2_base.mp3
   ♻️  Reutilizado desde Drive: audio_escena_2_base.mp3 (22.93s)

📢 [3/10] audio_escena_3_caso1.mp3
   ♻️  Reutilizado desde Drive: audio_escena_3_caso1.mp3 (20.53s)

📢 [4/10] audio_escena_4_caso2.mp3
   ♻️  Reutilizado desde Drive: audio_escena_4_caso2.mp3 (21.53s)

📢 [5/10] audio_escena_5_caso3.mp3
   ♻️  Reutilizado desde Drive: audio_escena_5_caso3.mp3 (20.77s)

📢 [6/10] audio_escena_6_caso4.mp3
   ♻️  Reutilizado desde Drive: audio_escena_6_caso4.mp3 (17.93s)

📢 [7/10] audio_escena_7_caso5.mp3
   ♻️  Reutilizado desde Drive: audio_escena_7_caso5.mp3 (17.93s)

📢 [8/10] audio_escena_8_caso6.mp3
   ♻️  Reutilizado desde Drive: audio_escena_8_caso6.mp3 (25.33s)

📢 [9/10] audio_escena_9_caso7.mp3
   ♻️  Reutilizado desde Drive: audio_escena_9_caso7.mp3 (25.97s)

In [4]:
import os, io, json, shutil
from pydub import AudioSegment

CARPETA_AUDIO = "/content/drive/MyDrive/manim_audios"

# Nombres reales de tus archivos
ARCHIVOS = [
    "audio_escena_1_intro.mp3",
    "audio_escena_2_base.mp3",
    "audio_escena_3_caso1.mp3",
    "audio_escena_4_caso2.mp3",
    "audio_escena_5_caso3.mp3",
    "audio_escena_6_caso4.mp3",
    "audio_escena_7_caso5.mp3",
    "audio_escena_8_caso6.mp3",
    "audio_escena_9_caso7.mp3",
    "audio_escena_10_cierre.mp3",
]

DURACIONES = {}

print("📂 Copiando audios de Drive al disco local...\n")
for nombre in ARCHIVOS:
    ruta_drive = os.path.join(CARPETA_AUDIO, nombre)
    ruta_local = f"/content/{nombre}"

    if os.path.exists(ruta_drive):
        shutil.copy(ruta_drive, ruta_local)
        seg = AudioSegment.from_file(ruta_drive, format="mp3")
        dur = len(seg) / 1000.0
        DURACIONES[nombre] = dur
        print(f"   ✅  {nombre:<35}  {dur:.2f}s")
    else:
        print(f"   ❌  NO ENCONTRADO en Drive: {nombre}")

# Exportar JSON para que Manim lo lea
json.dump(DURACIONES, open("duraciones.json", "w"), indent=2)

print(f"\n✅ duraciones.json guardado con {len(DURACIONES)} entradas")
total = sum(DURACIONES.values())
print(f"⏱️  Duración total estimada: {total:.1f}s ({total/60:.1f} min)")

📂 Copiando audios de Drive al disco local...

   ✅  audio_escena_1_intro.mp3             15.25s
   ✅  audio_escena_2_base.mp3              22.93s
   ✅  audio_escena_3_caso1.mp3             20.53s
   ✅  audio_escena_4_caso2.mp3             21.53s
   ✅  audio_escena_5_caso3.mp3             20.77s
   ✅  audio_escena_6_caso4.mp3             17.93s
   ✅  audio_escena_7_caso5.mp3             17.93s
   ✅  audio_escena_8_caso6.mp3             25.33s
   ✅  audio_escena_9_caso7.mp3             25.97s
   ✅  audio_escena_10_cierre.mp3           14.13s

✅ duraciones.json guardado con 10 entradas
⏱️  Duración total estimada: 202.3s (3.4 min)


In [5]:
# ==============================================================================
# CELDA 1: INSTALACIÓN DE LIBRERÍAS Y DEPENDENCIAS
# Ejecutar solo una vez. Puede tardar entre 1 y 2 minutos.
# ==============================================================================
import os
import subprocess
from IPython.display import clear_output

print("⚙️ Instalando Manim, dependencias de sistema y LaTeX...")

# Actualizar repositorios e instalar librerías multimedia y de tipografía
subprocess.run("apt-get update -qq", shell=True)
subprocess.run("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libcairo2-dev libffi-dev libpango1.0-dev ffmpeg texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended > /dev/null 2>&1", shell=True)

# Instalar el motor de Manim para Python
subprocess.run("pip install -q manim", shell=True)

# Limpiar pantalla y confirmar
clear_output()
print("✅ ¡Instalación completada con éxito!")
print("Ya puedes ejecutar la Celda 2 para crear la animación.")

✅ ¡Instalación completada con éxito!
Ya puedes ejecutar la Celda 2 para crear la animación.


In [1]:
from manim import *
import numpy as np
import json, os

DURACIONES = json.load(open("duraciones.json"))

def dur(nombre):
    return DURACIONES.get(nombre, 5.0)

In [2]:
%%manim -qm -v WARNING Transformaciones

NEON_CYAN   = "#00E5FF"
NEON_GREEN  = "#00FF9F"
NEON_PINK   = "#FF2D78"
NEON_YELLOW = "#FFD60A"
NEON_PURPLE = "#BF5FFF"
NEON_ORANGE = "#FF6B35"
NEON_LIME   = "#C6FF00"
DARK_PANEL  = "#0A1628"

CASE_COLORS = [
    NEON_GREEN, NEON_PINK, NEON_PURPLE,
    NEON_ORANGE, NEON_LIME, "#FF9F1C", "#FF2D9F",
]

class Transformaciones(Scene):
    def construct(self):

        PLANE_CENTER = LEFT  * 2.8
        PANEL_CENTER = RIGHT * 4.8
        PANEL_W      = 5.0
        PANEL_H      = 7.5

        # ══════════════════════════════════════════════════════════════
        # ESCENA 1 — INTRO
        # ══════════════════════════════════════════════════════════════
        self.add_sound("audio_escena_1_intro.mp3")

        intro_bg = RoundedRectangle(
            width=9.8, height=4.0, corner_radius=0.4,
            fill_color=DARK_PANEL, fill_opacity=0.97,
            stroke_color=NEON_CYAN, stroke_width=2.5,
        )
        intro_bar = RoundedRectangle(
            width=9.8, height=0.55, corner_radius=0.18,
            fill_color=NEON_CYAN, fill_opacity=1, stroke_width=0,
        ).next_to(intro_bg, UP, buff=0).shift(DOWN * 0.27)

        t_pre  = Text("TRANSFORMACIONES DE", font_size=24,
                      color=GREY_A, weight=BOLD)
        t_func = Text("y = e^x", font_size=78, color=NEON_CYAN)
        t_cred = Text("Docente: Wilmar Gonzalez  ·  Apoyado con IA",
                      font_size=17, color=GREY_B)
        intro_content = VGroup(t_pre, t_func, t_cred)\
                        .arrange(DOWN, buff=0.28).move_to(ORIGIN)

        ANIM_INTRO = 0.6 + 0.3 + 0.6 + 1.0 + 0.6
        self.play(FadeIn(intro_bg, scale=0.85), run_time=0.6)
        self.play(FadeIn(intro_bar), run_time=0.3)
        self.play(FadeIn(t_pre), run_time=0.6)
        self.play(Write(t_func), run_time=1.0)
        self.play(FadeIn(t_cred), run_time=0.6)
        self.wait(max(0.1, dur("audio_escena_1_intro.mp3") - ANIM_INTRO))
        self.play(FadeOut(VGroup(intro_bg, intro_bar, intro_content)),
                  run_time=0.5)

        # ══════════════════════════════════════════════════════════════
        # ESCENA 2 — PLANO + FUNCIÓN BASE
        # ══════════════════════════════════════════════════════════════
        self.add_sound("audio_escena_2_base.mp3")

        plane = NumberPlane(
            x_range=[-8, 8, 1], y_range=[-8, 8, 1],
            x_length=7, y_length=7,
            background_line_style={
                "stroke_color": "#1B3A6B",
                "stroke_width": 1.0,
                "stroke_opacity": 0.55,
            },
            axis_config={"stroke_color": "#3A7BD5", "stroke_width": 1.8},
        )
        plane.add_coordinates(font_size=15)
        plane.move_to(PLANE_CENTER)

        panel_bg = RoundedRectangle(
            width=PANEL_W, height=PANEL_H, corner_radius=0.4,
            fill_color=DARK_PANEL, fill_opacity=0.97,
            stroke_color="#1E3A5F", stroke_width=1.5,
        ).move_to(PANEL_CENTER)

        divider = Line(
            UP * 3.8, DOWN * 3.8,
            stroke_color=NEON_CYAN, stroke_width=0.8, stroke_opacity=0.25,
        ).move_to(RIGHT * 2.2)

        func_orig  = lambda x: np.exp(x)
        graph_glow = plane.plot(func_orig, x_range=[-8, 2.07],
                                color=NEON_CYAN,
                                stroke_width=10, stroke_opacity=0.10)
        graph_orig = plane.plot(func_orig, x_range=[-8, 2.07],
                                color=NEON_CYAN, stroke_width=2.8)
        eq_orig    = Text("y = e^x", font_size=44, color=NEON_CYAN)\
                     .move_to(PANEL_CENTER + UP * 0.1)

        dot_curve  = Dot(plane.c2p(0, 1), color=NEON_YELLOW, radius=0.12)
        lbl_curve  = Text("(0, 1)", font_size=26, color=NEON_YELLOW)\
                     .next_to(dot_curve, UL, buff=0.12)

        lbl_b_tit = Text("BASE", font_size=52, color=NEON_CYAN, weight=BOLD)\
                    .move_to(PANEL_CENTER + UP * 2.8)
        lbl_b_sub = Text("Funcion exponencial", font_size=20, color=WHITE)\
                    .move_to(PANEL_CENTER + UP * 1.9)
        lbl_dom   = Text("Dominio: todos los reales",
                         font_size=16, color=GREY_A)\
                    .move_to(PANEL_CENTER + DOWN * 0.8)
        lbl_ran   = Text("Rango: ( 0, +inf )",
                         font_size=16, color=GREY_A)\
                    .move_to(PANEL_CENTER + DOWN * 1.25)
        lbl_pt    = Text("Punto clave:", font_size=15, color=NEON_YELLOW)\
                    .move_to(PANEL_CENTER + DOWN * 2.0)
        lbl_01    = Text("(0, 1)", font_size=36, color=NEON_YELLOW)\
                    .move_to(PANEL_CENTER + DOWN * 2.6)
        base_panel = VGroup(lbl_b_tit, lbl_b_sub, lbl_dom,
                            lbl_ran, lbl_pt, lbl_01)

        ANIM_BASE = 1.5 + 1.8 + 0.8 + 0.8
        self.play(Create(plane, run_time=1.5),
                  FadeIn(panel_bg), FadeIn(divider))
        self.play(Create(graph_glow), Create(graph_orig),
                  Write(eq_orig), run_time=1.8)
        self.play(FadeIn(dot_curve, scale=1.8),
                  Write(lbl_curve), run_time=0.8)
        self.play(Write(lbl_b_tit), Write(lbl_b_sub),
                  FadeIn(lbl_dom), FadeIn(lbl_ran),
                  Write(lbl_pt), Write(lbl_01), run_time=0.8)
        self.wait(max(0.1, dur("audio_escena_2_base.mp3") - ANIM_BASE))
        self.play(FadeOut(base_panel))

        # ══════════════════════════════════════════════════════════════
        # FUNCIÓN AUXILIAR
        # ══════════════════════════════════════════════════════════════
        case_idx = [0]

        def apply_transform(
            caso, subtitulo, func_text, regla_text,
            target_func, target_x_range, p_curve_tgt,
            audio_file, extra_mobjects=None,
        ):
            color = CASE_COLORS[case_idx[0] % len(CASE_COLORS)]
            n     = case_idx[0] + 1
            case_idx[0] += 1
            nx, ny = p_curve_tgt

            self.add_sound(audio_file)

            top_bar = RoundedRectangle(
                width=PANEL_W - 0.06, height=0.52, corner_radius=0.18,
                fill_color=color, fill_opacity=1, stroke_width=0,
            ).move_to(PANEL_CENTER + UP * 3.38)
            badge = Text(f"Caso  {n} / 7", font_size=17,
                         color=DARK_PANEL, weight=BOLD)\
                    .move_to(top_bar.get_center())
            caso_lbl = Text(caso, font_size=55, color=color, weight=BOLD)\
                       .move_to(PANEL_CENTER + UP * 2.52)
            sub_lbl  = Text(subtitulo, font_size=19, color=WHITE,
                            line_spacing=1.25)\
                       .move_to(PANEL_CENTER + UP * 1.62)
            sep1 = Line(PANEL_CENTER+LEFT*2.1+UP*0.98,
                        PANEL_CENTER+RIGHT*2.1+UP*0.98,
                        stroke_color=color, stroke_width=1.2,
                        stroke_opacity=0.45)
            func_micro = Text("F U N C I O N", font_size=11,
                               color=color, weight=BOLD)\
                         .move_to(PANEL_CENTER + UP * 0.7)
            new_eq = Text(func_text, font_size=38, color=color)\
                     .move_to(PANEL_CENTER + UP * 0.05)
            sep2 = Line(PANEL_CENTER+LEFT*2.1+DOWN*0.54,
                        PANEL_CENTER+RIGHT*2.1+DOWN*0.54,
                        stroke_color=color, stroke_width=1.2,
                        stroke_opacity=0.45)
            regla_micro = Text("R E G L A", font_size=11,
                                color=color, weight=BOLD)\
                          .move_to(PANEL_CENTER + DOWN * 0.82)
            regla_val = Text(regla_text, font_size=16, color=GREY_A,
                             line_spacing=1.25)\
                        .move_to(PANEL_CENTER + DOWN * 1.46)
            sep3 = Line(PANEL_CENTER+LEFT*2.1+DOWN*2.18,
                        PANEL_CENTER+RIGHT*2.1+DOWN*2.18,
                        stroke_color=color, stroke_width=1.2,
                        stroke_opacity=0.45)
            punto_micro = Text("PUNTO (0,1) ->", font_size=11,
                                color=NEON_YELLOW, weight=BOLD)\
                          .move_to(PANEL_CENTER + DOWN * 2.58)
            punto_val = Text(f"({nx}, {ny})", font_size=38,
                             color=NEON_YELLOW)\
                        .move_to(PANEL_CENTER + DOWN * 3.18)

            panel_content = VGroup(
                top_bar, badge, caso_lbl, sub_lbl,
                sep1, func_micro, new_eq,
                sep2, regla_micro, regla_val,
                sep3, punto_micro, punto_val,
            )

            g_new_glow = plane.plot(target_func, x_range=target_x_range,
                                    color=color,
                                    stroke_width=10, stroke_opacity=0.10)
            g_new      = plane.plot(target_func, x_range=target_x_range,
                                    color=color, stroke_width=2.8)
            new_dot    = Dot(plane.c2p(*p_curve_tgt),
                             color=NEON_YELLOW, radius=0.12)
            new_lbl    = Text(f"({nx}, {ny})", font_size=26,
                              color=NEON_YELLOW)\
                         .next_to(new_dot, UL, buff=0.12)

            RT_BADGE     = 0.5
            RT_TITULO    = 0.9
            RT_EXTRAS    = 0.9 if extra_mobjects else 0.0
            RT_TRANSFORM = 2.2
            RT_PANEL     = 0.4 + 0.8 + 0.6
            RT_FLASH     = 0.6
            ANIM_TOTAL   = (RT_BADGE + RT_TITULO + RT_EXTRAS +
                            RT_TRANSFORM + RT_PANEL + RT_FLASH)

            self.play(FadeIn(top_bar), Write(badge), run_time=RT_BADGE)
            self.play(Write(caso_lbl), Write(sub_lbl), run_time=RT_TITULO)

            if extra_mobjects:
                self.play(*[Create(m) for m in extra_mobjects],
                          run_time=RT_EXTRAS)

            self.play(
                Transform(graph_glow, g_new_glow),
                Transform(graph_orig, g_new),
                Transform(eq_orig,    new_eq),
                dot_curve.animate.move_to(new_dot.get_center()),
                Transform(lbl_curve,  new_lbl),
                run_time=RT_TRANSFORM,
            )
            self.play(Create(sep1), Write(func_micro), run_time=0.4)
            self.play(Create(sep2), Write(regla_micro),
                      Write(regla_val), run_time=0.8)
            self.play(Create(sep3), Write(punto_micro),
                      Write(punto_val), run_time=0.6)
            self.play(Flash(dot_curve, color=NEON_YELLOW,
                            line_length=0.20, num_lines=10,
                            flash_radius=0.32, run_time=RT_FLASH))

            self.wait(max(0.3, dur(audio_file) - ANIM_TOTAL))

            g_base_r  = plane.plot(func_orig, x_range=[-8, 2.07],
                                   color=NEON_CYAN, stroke_width=2.8)
            g_glow_r  = plane.plot(func_orig, x_range=[-8, 2.07],
                                   color=NEON_CYAN,
                                   stroke_width=10, stroke_opacity=0.10)
            eq_base_r = Text("y = e^x", font_size=44, color=NEON_CYAN)\
                        .move_to(PANEL_CENTER + UP * 0.1)
            lbl_r     = Text("(0, 1)", font_size=26, color=NEON_YELLOW)\
                        .next_to(plane.c2p(0, 1), UL, buff=0.12)

            revert = [
                Transform(graph_glow, g_glow_r),
                Transform(graph_orig, g_base_r),
                Transform(eq_orig,    eq_base_r),
                dot_curve.animate.move_to(plane.c2p(0, 1)),
                Transform(lbl_curve,  lbl_r),
                FadeOut(panel_content),
            ]
            if extra_mobjects:
                revert.extend([FadeOut(m) for m in extra_mobjects])

            self.play(*revert, run_time=1.5)
            self.wait(0.3)

        # ══════════════════════════════════════════════════════════════
        # CASOS (a) → (g)
        # ══════════════════════════════════════════════════════════════
        apply_transform(
            caso="(a)", subtitulo="Desplazar 2 unidades\nhacia abajo",
            func_text="y = e^x - 2",
            regla_text="f(x) - k  baja k unidades",
            target_func=lambda x: np.exp(x) - 2,
            target_x_range=[-8, 2.3], p_curve_tgt=(0, -1),
            audio_file="audio_escena_3_caso1.mp3",
        )
        apply_transform(
            caso="(b)", subtitulo="Desplazar 2 unidades\nhacia la derecha",
            func_text="y = e^(x-2)",
            regla_text="f(x-h)  mueve h unidades\na la derecha",
            target_func=lambda x: np.exp(x - 2),
            target_x_range=[-6, 4.07], p_curve_tgt=(2, 1),
            audio_file="audio_escena_4_caso2.mp3",
        )
        apply_transform(
            caso="(c)", subtitulo="Reflejar sobre\nel eje x",
            func_text="y = -e^x",
            regla_text="-f(x)  refleja sobre el eje x",
            target_func=lambda x: -np.exp(x),
            target_x_range=[-8, 2.07], p_curve_tgt=(0, -1),
            audio_file="audio_escena_5_caso3.mp3",
        )
        apply_transform(
            caso="(d)", subtitulo="Reflejar sobre\nel eje y",
            func_text="y = e^(-x)",
            regla_text="f(-x)  refleja sobre el eje y",
            target_func=lambda x: np.exp(-x),
            target_x_range=[-2.07, 8], p_curve_tgt=(0, 1),
            audio_file="audio_escena_6_caso4.mp3",
        )
        apply_transform(
            caso="(e)", subtitulo="Reflejar eje x\ny luego eje y",
            func_text="y = -e^(-x)",
            regla_text="-f(-x)  doble reflexion",
            target_func=lambda x: -np.exp(-x),
            target_x_range=[-2.07, 8], p_curve_tgt=(0, -1),
            audio_file="audio_escena_7_caso5.mp3",
        )

        line_y4  = DashedLine(plane.c2p(-8, 4), plane.c2p(8, 4),
                              color=WHITE, stroke_width=1.8,
                              dash_length=0.22, stroke_opacity=0.65)
        label_y4 = Text("y = 4", font_size=24, color=WHITE)\
                   .next_to(plane.c2p(-5, 4), UP, buff=0.1)
        apply_transform(
            caso="(f)", subtitulo="Reflejar sobre\nla recta  y = 4",
            func_text="y = 8 - e^x",
            regla_text="y' = 2(4) - e^x = 8 - e^x",
            target_func=lambda x: 8 - np.exp(x),
            target_x_range=[-8, 2.77], p_curve_tgt=(0, 7),
            audio_file="audio_escena_8_caso6.mp3",
            extra_mobjects=[line_y4, label_y4],
        )

        line_x2  = DashedLine(plane.c2p(2, -8), plane.c2p(2, 8),
                              color=WHITE, stroke_width=1.8,
                              dash_length=0.22, stroke_opacity=0.65)
        label_x2 = Text("x = 2", font_size=24, color=WHITE)\
                   .next_to(plane.c2p(2, 5), UR, buff=0.1)
        apply_transform(
            caso="(g)", subtitulo="Reflejar sobre\nla recta  x = 2",
            func_text="y = e^(4-x)",
            regla_text="x' = 4 - x  ->  y = e^(4-x)",
            target_func=lambda x: np.exp(4 - x),
            target_x_range=[1.92, 8], p_curve_tgt=(4, 1),
            audio_file="audio_escena_9_caso7.mp3",
            extra_mobjects=[line_x2, label_x2],
        )

        # ══════════════════════════════════════════════════════════════
        # CIERRE
        # ══════════════════════════════════════════════════════════════
        self.play(*[FadeOut(m) for m in self.mobjects])
        self.add_sound("audio_escena_10_cierre.mp3")

        final_bg  = RoundedRectangle(
            width=10.2, height=3.2, corner_radius=0.4,
            fill_color=DARK_PANEL, fill_opacity=0.97,
            stroke_color=NEON_CYAN, stroke_width=2.5)
        final_bar = RoundedRectangle(
            width=10.2, height=0.52, corner_radius=0.18,
            fill_color=NEON_CYAN, fill_opacity=1, stroke_width=0)\
            .next_to(final_bg, UP, buff=0).shift(DOWN * 0.26)
        f1 = Text("Transformaciones completadas",
                  font_size=36, color=NEON_CYAN, weight=BOLD)
        f2 = Text("Docente: Wilmar Gonzalez",
                  font_size=22, color=GREY_A)
        VGroup(f1, f2).arrange(DOWN, buff=0.30).move_to(ORIGIN)

        ANIM_CIERRE = 0.7 + 0.8
        self.play(FadeIn(final_bg, scale=0.85),
                  FadeIn(final_bar), run_time=0.7)
        self.play(Write(f1), FadeIn(f2), run_time=0.8)
        self.wait(max(0.3,
                      dur("audio_escena_10_cierre.mp3") - ANIM_CIERRE))
        self.play(FadeOut(VGroup(final_bg, final_bar, f1, f2)))

Output hidden; open in https://colab.research.google.com to view.

In [3]:
from google.colab import files
import glob
import os

# Buscamos el video generado
videos = glob.glob("media/**/*.mp4", recursive=True)

if videos:
    video_final = max(videos, key=os.path.getctime)
    print("¡Video encontrado! Iniciando descarga de tu animación con audio...")
    files.download(video_final)
else:
    print("Hubo un error, no se encontró el video.")

¡Video encontrado! Iniciando descarga de tu animación con audio...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>